# Model 6 — Feature Association Rule Mining (Apriori)

## Research Question
What combinations of car features tend to appear together — and do certain combinations reliably predict very expensive, very cheap, or heavily damaged cars?

## Introduction
This notebook uses the **Apriori algorithm** (`mlxtend`) to discover association rules between binary car features.

**Rules for this notebook:**
- Uses **unscaled data** (`proceed_dataset_without_scaling.csv`)
- You **must use `apriori` from `mlxtend`** — no other algorithm is acceptable
- You **must engineer the binary columns** described in the Feature Engineering section before running Apriori
- You **may adjust `min_support` and `min_confidence` thresholds** to explore different rule sets
- You **may choose different features**, add features — but you **cannot change the general technique category** (must remain Apriori / association rule mining)

## Data Import

Run this cell as-is. It loads all required libraries and the unscaled dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules

df = pd.read_csv('../../data/proceed_dataset_without_scaling.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## Feature Engineering

Run this cell as-is. It creates seven custom binary columns that will be used as special items in the transaction matrix:

| Column | Meaning |
|---|---|
| `cok_pahali` | Price ≥ 95th percentile |
| `cok_ucuz` | Price ≤ 5th percentile |
| `yuksek_km` | Mileage > 150,000 km |
| `dusuk_km` | Mileage < 30,000 km |
| `yeni_araba` | Year ≥ 2022 |
| `eski_araba` | Year ≤ 2015 |
| `agir_hasar` | Damage score ≥ 75th percentile |

In [ ]:
# Create custom binary columns for association mining
df['cok_pahali'] = (df['Fiyat'] >= df['Fiyat'].quantile(0.95)).astype(int)
df['cok_ucuz']   = (df['Fiyat'] <= df['Fiyat'].quantile(0.05)).astype(int)
df['yuksek_km']  = (df['Kilometre'] > 150000).astype(int)
df['dusuk_km']   = (df['Kilometre'] < 30000).astype(int)
df['yeni_araba'] = (df['Yıl'] >= 2022).astype(int)
df['eski_araba'] = (df['Yıl'] <= 2015).astype(int)
df['agir_hasar'] = (df['paint_damage_score'] >= df['paint_damage_score'].quantile(0.75)).astype(int)

print("Engineered columns created:")
for col in ['cok_pahali', 'cok_ucuz', 'yuksek_km', 'dusuk_km', 'yeni_araba', 'eski_araba', 'agir_hasar']:
    print(f"  {col}: {df[col].sum()} cars ({df[col].mean()*100:.1f}%)")

## Transaction Dataset Construction

Run this cell as-is. It combines the engineered binary columns with all existing one-hot encoded columns (body type, fuel type, drivetrain, seller type, brand flags) into a boolean transaction matrix suitable for Apriori.

In [ ]:
# Combine engineered columns with existing binary one-hot columns
engineered_cols = ['cok_pahali', 'cok_ucuz', 'yuksek_km', 'dusuk_km',
                   'yeni_araba', 'eski_araba', 'agir_hasar']

# Find all existing binary one-hot columns
onehot_patterns = ['Kasa Tipi_', 'Yakıt Tipi_', 'Çekiş_', 'Kimden_', 'is_Nissan']
onehot_cols = [c for c in df.columns for p in onehot_patterns if c.startswith(p)]

transaction_cols = engineered_cols + onehot_cols
transaction_cols = [c for c in transaction_cols if c in df.columns]

transactions = df[transaction_cols].fillna(0).astype(bool)
print(f"Transaction matrix: {transactions.shape[0]} rows × {transactions.shape[1]} items")
print(f"Total item columns: {transactions.columns.tolist()}")

## Frequent Itemset Mining

**TODO (Student Task):** Run the Apriori algorithm to find frequent itemsets.

- Lower `min_support` → more itemsets found, but may include noisy/spurious ones
- Higher `min_support` → fewer, more reliable itemsets
- Adjust `max_len` to control the maximum number of items per itemset

In [ ]:
# TODO: Adjust min_support. Lower values find more rules but may be noisy.
# Recommended starting point: 0.05 (5% of transactions)
frequent_itemsets = apriori(
    transactions,
    min_support=0.05,      # TODO: try 0.03, 0.08, 0.10
    use_colnames=True,
    max_len=4              # TODO: max items per itemset
)
print(f"Found {len(frequent_itemsets)} frequent itemsets")
frequent_itemsets.sort_values('support', ascending=False).head(20)

## Association Rules Generation

**TODO (Student Task):** Generate association rules from the frequent itemsets.

- Higher `min_threshold` for confidence → fewer but more reliable rules
- Sort by `lift` to find rules where the co-occurrence is much higher than chance

In [ ]:
# TODO: Adjust min_threshold for confidence. Higher = fewer but more reliable rules.
rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.5      # TODO: try 0.6, 0.7
)
rules = rules.sort_values('lift', ascending=False)
print(f"Total rules generated: {len(rules)}")
rules.head(10)

## Evaluation

### Top Rules Predicting Very Expensive Cars (`cok_pahali`)

Filters rules whose consequent is `cok_pahali` and ranks them by confidence.

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual association rules after running Apriori.

In [ ]:
# ⚠️ Replace rules with your actual association rules after running apriori.
pahali_rules = rules[rules['consequents'].apply(lambda x: 'cok_pahali' in x)]
pahali_rules_sorted = pahali_rules.sort_values('confidence', ascending=False).head(10)

print("=== Top 10 Rules Predicting VERY EXPENSIVE Cars ===\n")
display_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
if len(pahali_rules_sorted) > 0:
    styled = pahali_rules_sorted[display_cols].reset_index(drop=True)
    styled['antecedents'] = styled['antecedents'].apply(lambda x: ', '.join(list(x)))
    styled['consequents'] = styled['consequents'].apply(lambda x: ', '.join(list(x)))
    print(styled.to_string(index=False))
    strong = pahali_rules_sorted[pahali_rules_sorted['lift'] > 2.0]
    if len(strong) > 0:
        print(f"\n⭐ {len(strong)} rule(s) with lift > 2.0 (STRONG associations):")
        for _, r in strong.iterrows():
            print(f"   {set(r['antecedents'])} → {set(r['consequents'])} | lift={r['lift']:.2f}")
else:
    print("No rules found. Try lowering min_support or min_threshold.")

### Top Rules Predicting Very Cheap Cars (`cok_ucuz`)

Filters rules whose consequent is `cok_ucuz` and ranks them by confidence.

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual association rules.

In [ ]:
# ⚠️ Replace rules with your actual association rules.
ucuz_rules = rules[rules['consequents'].apply(lambda x: 'cok_ucuz' in x)]
ucuz_rules_sorted = ucuz_rules.sort_values('confidence', ascending=False).head(10)

print("=== Top 10 Rules Predicting VERY CHEAP Cars ===\n")
if len(ucuz_rules_sorted) > 0:
    styled = ucuz_rules_sorted[display_cols].reset_index(drop=True).copy()
    styled['antecedents'] = styled['antecedents'].apply(lambda x: ', '.join(list(x)))
    styled['consequents'] = styled['consequents'].apply(lambda x: ', '.join(list(x)))
    print(styled.to_string(index=False))
else:
    print("No rules found. Try lowering min_support or min_threshold.")

### Top Rules Predicting Heavily Damaged Cars (`agir_hasar`)

Filters rules whose consequent is `agir_hasar` and ranks them by confidence.

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual association rules.

In [ ]:
# ⚠️ Replace rules with your actual association rules.
hasar_rules = rules[rules['consequents'].apply(lambda x: 'agir_hasar' in x)]
hasar_rules_sorted = hasar_rules.sort_values('confidence', ascending=False).head(10)

print("=== Top 10 Rules Predicting HEAVILY DAMAGED Cars ===\n")
if len(hasar_rules_sorted) > 0:
    styled = hasar_rules_sorted[display_cols].reset_index(drop=True).copy()
    styled['antecedents'] = styled['antecedents'].apply(lambda x: ', '.join(list(x)))
    styled['consequents'] = styled['consequents'].apply(lambda x: ', '.join(list(x)))
    print(styled.to_string(index=False))
else:
    print("No rules found. Try lowering min_support or min_threshold.")

### Top 10 Rules by Lift (Strongest Natural Associations)

Lift > 1 means the items co-occur more than expected by chance. Lift > 2 is considered a strong association.

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual association rules.

In [ ]:
# ⚠️ Replace rules with your actual association rules.
top_lift = rules.sort_values('lift', ascending=False).head(10)
print("=== Top 10 Rules by Lift (Strongest Natural Associations) ===\n")
styled = top_lift[display_cols].reset_index(drop=True).copy()
styled['antecedents'] = styled['antecedents'].apply(lambda x: ', '.join(list(x)))
styled['consequents'] = styled['consequents'].apply(lambda x: ', '.join(list(x)))
print(styled.to_string(index=False))

strong_all = rules[rules['lift'] > 2.0]
print(f"\n⭐ Total rules with lift > 2.0: {len(strong_all)}")

### Support vs Confidence Scatter Plot

Each point is one association rule. Point size encodes lift. Color indicates the type of consequent:
- 🔴 Red = `cok_pahali` (very expensive)
- 🔵 Blue = `cok_ucuz` (very cheap)
- 🟣 Purple = `agir_hasar` (heavy damage)
- ⚫ Grey = other consequents

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual association rules.

In [ ]:
# ⚠️ Replace rules with your actual association rules.
target_consequents = ['cok_pahali', 'cok_ucuz', 'agir_hasar']
color_map = {'cok_pahali': '#e74c3c', 'cok_ucuz': '#3498db', 'agir_hasar': '#8e44ad', 'other': '#bdc3c7'}

def get_color(row):
    for t in target_consequents:
        if t in row['consequents']:
            return color_map[t]
    return color_map['other']

plot_rules = rules.copy()
plot_rules['color'] = plot_rules.apply(get_color, axis=1)
plot_rules['lift_scaled'] = (plot_rules['lift'] - plot_rules['lift'].min()) / (plot_rules['lift'].max() - plot_rules['lift'].min() + 1e-9) * 200 + 20

fig, ax = plt.subplots(figsize=(11, 7))
for label, color in color_map.items():
    mask = plot_rules['color'] == color
    if mask.sum() > 0:
        ax.scatter(plot_rules.loc[mask, 'support'],
                   plot_rules.loc[mask, 'confidence'],
                   s=plot_rules.loc[mask, 'lift_scaled'],
                   c=color, alpha=0.6, label=label, edgecolors='none')

ax.set_xlabel('Support', fontsize=13)
ax.set_ylabel('Confidence', fontsize=13)
ax.set_title('Association Rules: Support vs Confidence\n(Point size = Lift)', fontsize=14)
ax.legend(title='Consequent Type')
plt.tight_layout()
plt.show()

### Top 10 Rules by Lift — Bar Chart

Horizontal bar chart of the 10 rules with the highest lift. Rules with lift > 2.0 are highlighted in red.

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual association rules.

In [ ]:
# ⚠️ Replace rules with your actual association rules.
top10_lift = rules.nlargest(10, 'lift').reset_index(drop=True)
top10_lift['rule_label'] = top10_lift.apply(
    lambda r: f"{', '.join(list(r['antecedents']))} → {', '.join(list(r['consequents']))}", axis=1)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#e74c3c' if r['lift'] > 2 else '#3498db' for _, r in top10_lift.iterrows()]
ax.barh(top10_lift['rule_label'], top10_lift['lift'], color=colors, edgecolor='white')
ax.axvline(2.0, color='black', linestyle='--', linewidth=1.5, label='Lift = 2.0 (Strong Association)')
ax.set_xlabel('Lift', fontsize=13)
ax.set_title('Top 10 Association Rules by Lift', fontsize=14)
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## ⚠️ If Your Model Underperforms

If your model produces poor or surprising results (e.g., very low accuracy, unexpected associations, or trivial rules), **do not discard your results**.

- Keep all outputs as-is
- In your presentation, document exactly what you observe
- Write a short hypothesis: Why might the model have failed? (e.g., 'The dataset may not contain enough transactions with rare feature combinations to produce reliable high-confidence rules')